# DATN · Trình diễn Gradio trên Colab
Notebook tùy chọn, chỉ suy luận và Replay. Không train model. Dùng Python 3.10 riêng, Xvfb cho render thật, GPU cho torch nếu có.
Upload `DATN-demo-source.zip` từ `git archive --format=zip --output=DATN-demo-source.zip Bao` của nhánh đã commit giao diện. File checkpoint giữ ngoài Git. Sửa hai đường dẫn artifact bên dưới cho đúng Drive hiện tại.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, zipfile, hashlib, re
drive.mount('/content/drive')
SOURCE_ZIP = Path('/content/DATN-demo-source.zip')  # Upload bằng sidebar Files trước
CHECKPOINT_DIR = Path('/content/drive/MyDrive/DATN_ARTIFACTS/checkpoints')  # Sửa theo vị trí thật
OUTPUT_DIR = Path('/content/drive/MyDrive/DATN_DEMO_OUTPUT')  # Thư mục output riêng
assert SOURCE_ZIP.is_file(), 'Cần upload source ZIP từ commit có demo_ui/app.py'
assert CHECKPOINT_DIR.is_dir(), 'Sửa CHECKPOINT_DIR cho đúng thư mục checkpoint'
ROOT = Path('/content/datn-demo')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as bundle:
    for item in bundle.infolist():
        assert (ROOT / item.filename).resolve().is_relative_to(ROOT.resolve()), 'ZIP chứa đường dẫn không hợp lệ'
    bundle.extractall(ROOT)
    SOURCE_COMMIT = bundle.comment.decode('ascii', errors='ignore') or 'source-zip'
assert (ROOT / 'demo_ui/app.py').is_file(), 'Source ZIP chưa có giao diện'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_SHA = hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest()
print('Source:', SOURCE_COMMIT, SOURCE_SHA)


In [ ]:
def run(*args):
    subprocess.run(list(args), check=True, cwd=ROOT)
run(sys.executable, '-m', 'pip', 'install', '-q', 'uv')
run('uv', 'python', 'install', '3.10')
VENV = Path('/content/datn-demo-venv')
if not (VENV / 'bin/python').exists():
    run('uv', 'venv', '--python', '3.10', str(VENV))
PYTHON = str(VENV / 'bin/python')
run('uv', 'pip', 'install', '--python', PYTHON, 'torch', '--index-url', 'https://download.pytorch.org/whl/cu128')
run('uv', 'pip', 'install', '--python', PYTHON, '-r', 'requirements-ui.txt', 'metadrive-simulator==0.4.3', 'gymnasium', 'tqdm')
run('apt-get', 'update', '-qq')
run('apt-get', 'install', '-y', '-qq', 'xvfb', 'xauth', 'libgl1', 'libegl1', 'libglib2.0-0')
run(PYTHON, '-c', 'import torch, gradio; print(torch.__version__, torch.cuda.is_available(), gradio.__version__)')
ENV = dict(os.environ, SDL_VIDEODRIVER='dummy', LIBGL_ALWAYS_SOFTWARE='1', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', DATN_SOURCE_COMMIT=SOURCE_COMMIT, DATN_SOURCE_ARCHIVE_SHA256=SOURCE_SHA)


## Kiểm tra render thật trước
Chọn đúng file đã có trong checkpoint directory. Ví dụ dưới dành cho c23 embedded WM. Nếu model chỉ chứa Actor cần thêm `--wm-checkpoint`. Xvfb dùng software OpenGL nên có thể chậm, CUDA vẫn dùng cho suy luận. Không thay driver NVIDIA.

In [ ]:
CHECKPOINT = 'critic_warmup/continuous_seed0_refit_warmup_c23.pt'  # Sửa nếu cần
subprocess.run(['xvfb-run', '-a', PYTHON, 'scripts/check_demo_runtime.py', '--checkpoint-dir', str(CHECKPOINT_DIR), '--checkpoint', CHECKPOINT, '--device', 'cuda', '--steps', '3', '--output-dir', str(OUTPUT_DIR / 'render_check')], cwd=ROOT, env=ENV, check=True)


In [ ]:
import time
LOG = OUTPUT_DIR / 'gradio.log'
if 'ui_process' in globals() and ui_process.poll() is None:
    raise RuntimeError('UI đang chạy. Dừng bằng cell cuối trước khi mở lại.')
with LOG.open('w') as log:
    ui_process = subprocess.Popen(['xvfb-run', '-a', PYTHON, '-u', 'scripts/demo_ui.py', '--checkpoint-dir', str(CHECKPOINT_DIR), '--output-dir', str(OUTPUT_DIR / 'episodes'), '--share', '--host', '0.0.0.0'], cwd=ROOT, env=ENV, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    text = LOG.read_text(errors='replace')
    links = re.findall(r'https://[a-zA-Z0-9.-]+\.gradio\.live', text)
    if links or ui_process.poll() is not None:
        print(text[-4000:])
        break
    time.sleep(1)
else:
    print('Xem log tại', LOG, 'để lấy link hoặc chẩn đoán.')


Mở link, chọn checkpoint, chọn `cuda`, nạp model và khởi tạo episode. Bản ghi nằm trên Drive theo từng bước. Với chế độ chỉ CPU, dùng `--replay-only`, bỏ bước cài torch/MetaDrive và kiểm render.

In [ ]:
import signal
if 'ui_process' in globals() and ui_process.poll() is None:
    os.killpg(ui_process.pid, signal.SIGTERM)
    try:
        ui_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        os.killpg(ui_process.pid, signal.SIGKILL)
print('Đã dừng process UI. Output đã nằm trên Drive. Chọn Disconnect and delete runtime để ngắt GPU Colab.')
